In [1]:
import numpy as np
from scipy.stats import f_oneway, f, tukey_hsd

<div style="background-color:#eef3fb; border-left:5px solid #4a7abf; padding:14px 18px; border-radius:6px; margin-bottom:8px;">

📌 **Question 1 : One-Way ANOVA (Fertilizer Yield)**

Three fertilizers tested on crop yield (tons/acre):

- A: `[5.2, 5.8, 5.5, 6.0, 5.7]`
- B: `[6.1, 6.4, 6.2, 6.8, 6.3]`
- C: `[4.8, 5.0, 4.9, 5.3, 5.1]`

Run one-way ANOVA at α = 0.05. Compute SS_between, SS_within, MS, F, and state your conclusion. Compute η² as well.

</div>

#### Step 1 : Hypotheses

Let $\mu_A, \mu_B, \mu_C$ = true mean crop yields for fertilizers A, B, and C.

$$H_0: \mu_A = \mu_B = \mu_C \quad \text{(all group means are equal)}$$
$$H_1: \text{at least one mean differs}$$

#### Step 2 : Why This Test

*We have three independent groups and want to compare their means simultaneously. Running multiple t-tests would inflate the Type I error rate, so one-way ANOVA is the right tool. It partitions total variability into between-group variance (signal) and within-group variance (noise), and compares them via the F ratio. Population variances are assumed equal across groups (homoscedasticity).*

#### Step 3 : Test Statistic

$$F = \frac{MS_{\text{between}}}{MS_{\text{within}}}, \qquad MS = \frac{SS}{df}$$

$$SS_{\text{between}} = \sum_{j} n_j (\bar{x}_j - \bar{x}_{\text{grand}})^2 \qquad SS_{\text{within}} = \sum_{j} \sum_{i} (x_{ij} - \bar{x}_j)^2$$

In [2]:
A = np.array([5.2, 5.8, 5.5, 6.0, 5.7])
B = np.array([6.1, 6.4, 6.2, 6.8, 6.3])
C = np.array([4.8, 5.0, 4.9, 5.3, 5.1])

alpha  = 0.05
groups = [A, B, C]
k      = len(groups)
n      = len(A)
N      = k * n

In [3]:
mu_A = np.mean(A)
mu_B = np.mean(B)
mu_C = np.mean(C)
mu   = np.mean(np.concatenate(groups))

print(f'mu_A : {mu_A:.4f}')
print(f'mu_B : {mu_B:.4f}')
print(f'mu_C : {mu_C:.4f}')
print(f'mu   : {mu:.4f}')

mu_A : 5.6400
mu_B : 6.3600
mu_C : 5.0200
mu   : 5.6733


In [4]:
means     = np.array([mu_A, mu_B, mu_C])
ss_bw     = n * np.sum((means - mu)**2)
ss_within = sum(np.sum((g - g.mean())**2) for g in groups)
ss_total  = ss_bw + ss_within

print(f'ss_bw    : {ss_bw:.4f}')
print(f'ss_within: {ss_within:.4f}')
print(f'ss_total : {ss_total:.4f}')

ss_bw    : 4.4973
ss_within: 0.8120
ss_total : 5.3093


In [5]:
df_bw     = k - 1
df_within = N - k

ms_bw     = ss_bw / df_bw
ms_within = ss_within / df_within
f_stat    = ms_bw / ms_within

print(f'df_bw    : {df_bw}')
print(f'df_within: {df_within}')
print(f'ms_bw    : {ms_bw:.4f}')
print(f'ms_within: {ms_within:.4f}')
print(f'f_stat   : {f_stat:.4f}')

df_bw    : 2
df_within: 12
ms_bw    : 2.2487
ms_within: 0.0677
f_stat   : 33.2315


#### Step 4 : Critical Value Method

In [6]:
f_crit = f.ppf(1 - alpha, df_bw, df_within)

print(f'f_stat : {f_stat:.4f}')
print(f'f_crit : {f_crit:.4f}')

f_stat : 33.2315
f_crit : 3.8853


**F_stat (33.2315) > F_crit (3.8853) → Reject H₀**

#### Step 5 : p-value Method

In [7]:
pval = f.sf(f_stat, df_bw, df_within)
print(f'p_val : {pval:.8f}')

p_val : 0.00001280


**p_val (0.00001280) < 0.05 → Reject H₀**

#### Step 6 : Final Conclusion

*η² (eta-squared) measures what proportion of total variance is explained by the group factor. It is the ANOVA equivalent of R². Values above 0.14 are conventionally considered large.*

$$\eta^2 = \frac{SS_{\text{between}}}{SS_{\text{total}}}$$

In [8]:
eta_sq = ss_bw / ss_total
print(f'eta_sq : {eta_sq:.4f}')

eta_sq : 0.8471


In [9]:
f_stat_, pval_ = f_oneway(A, B, C)
print(f'f_stat (scipy) : {f_stat_:.4f}')
print(f'p_val  (scipy) : {pval_:.8f}')

f_stat (scipy) : 33.2315
p_val  (scipy) : 0.00001280


#### ✅ Final Conclusion

| Metric | Value |
|---|---|
| Test statistic (F) | 33.2315 |
| Critical value | 3.8853 |
| p-value | 0.00001280 |
| df (between, within) | 2, 12 |
| α | 0.05 |
| η² | 0.8471 |
| Decision | **Reject H₀** |

Strong evidence that at least one fertilizer produces a different mean crop yield. The F statistic (33.23) far exceeds the critical value (3.89), and the p-value is well below 0.05. η² = 0.8471 means the fertilizer choice explains about 84.7% of the total variance in yield, a very large effect. Fertilizer B consistently outperforms both A and C, while C yields the least.

<div style="background-color:#eef3fb; border-left:5px solid #4a7abf; padding:14px 18px; border-radius:6px; margin-bottom:8px;">

📌 **Question 2 : ANOVA Table Interpretation**

Given: SS_between = 120, SS_within = 180, k = 4 groups, N = 24 total observations.

- (a) Fill in the ANOVA table: df, MS_between, MS_within, F.
- (b) Is the result significant at α = 0.05?
- (c) Compute η².

</div>

#### Step 1 : Hypotheses

Let $\mu_1, \mu_2, \mu_3, \mu_4$ = true means for the four groups.

$$H_0: \mu_1 = \mu_2 = \mu_3 = \mu_4 \quad \text{(all group means are equal)}$$
$$H_1: \text{at least one mean differs}$$

#### Step 2 : Why This Test

*The SS values are given directly rather than raw data, so we reconstruct the ANOVA table from first principles. The logic is identical to Question 1: we compare between-group variance to within-group variance via the F ratio, using the same assumptions of independence, normality, and equal variances.*

#### Step 3 : Test Statistic

$$F = \frac{MS_{\text{between}}}{MS_{\text{within}}}, \qquad MS_{\text{between}} = \frac{SS_{\text{between}}}{k - 1}, \qquad MS_{\text{within}} = \frac{SS_{\text{within}}}{N - k}$$

In [10]:
ss_bw     = 120
ss_within = 180
k         = 4
N         = 24
alpha     = 0.05

In [11]:
df_bw     = k - 1
df_within = N - k
ss_total  = ss_bw + ss_within

ms_bw     = ss_bw / df_bw
ms_within = ss_within / df_within
f_stat    = ms_bw / ms_within

print(f'df_bw    : {df_bw}')
print(f'df_within: {df_within}')
print(f'ms_bw    : {ms_bw:.4f}')
print(f'ms_within: {ms_within:.4f}')
print(f'f_stat   : {f_stat:.4f}')

df_bw    : 3
df_within: 20
ms_bw    : 40.0000
ms_within: 9.0000
f_stat   : 4.4444


#### Step 4 : Critical Value Method

In [12]:
f_crit = f.ppf(1 - alpha, df_bw, df_within)

print(f'f_stat : {f_stat:.4f}')
print(f'f_crit : {f_crit:.4f}')

f_stat : 4.4444
f_crit : 3.0984


**F_stat (4.4444) > F_crit (3.0984) → Reject H₀**

#### Step 5 : p-value Method

In [13]:
pval = f.sf(f_stat, df_bw, df_within)
print(f'p_val : {pval:.6f}')

p_val : 0.015063


**p_val (0.015063) < 0.05 → Reject H₀**

#### Step 6 : Final Conclusion

*η² quantifies the proportion of total variance attributable to group differences.*

$$\eta^2 = \frac{SS_{\text{between}}}{SS_{\text{total}}}$$

In [14]:
eta_sq = ss_bw / ss_total
print(f'eta_sq : {eta_sq:.4f}')

eta_sq : 0.4000


In [15]:
# No raw data available for scipy verification in this question.
# The F statistic was derived directly from the given SS values.

#### ✅ Final Conclusion

| Metric | Value |
|---|---|
| Test statistic (F) | 4.4444 |
| Critical value | 3.0984 |
| p-value | 0.015063 |
| df (between, within) | 3, 20 |
| α | 0.05 |
| η² | 0.4000 |
| Decision | **Reject H₀** |

The result is significant at α = 0.05. F (4.44) exceeds the critical value (3.10) and the p-value (0.015) falls below the threshold, so we reject the null hypothesis. η² = 0.40 indicates that group membership explains 40% of the total variance, a large effect by conventional standards.

<div style="background-color:#eef3fb; border-left:5px solid #4a7abf; padding:14px 18px; border-radius:6px; margin-bottom:8px;">

📌 **Question 3 : One-Way ANOVA (ML Training Methods)**

A researcher compares the effectiveness of 5 machine-learning training methods on model accuracy scores (%). The observations are:

- Method A: `[78, 82, 75, 80, 79, 81]`
- Method B: `[91, 88, 85, 90, 87, 89]`
- Method C: `[72, 70, 74, 73, 71, 69]`
- Method D: `[95, 92, 96, 94, 91, 93]`
- Method E: `[84, 83, 82, 81, 85, 86]`

Perform a complete one-way ANOVA at α = 0.01.

- (a) Compute group means, grand mean, SSB, SSW, SST.
- (b) Construct the complete ANOVA table.
- (c) Test significance using both critical value and p-value approaches.
- (d) Compute η² and interpret the effect size.
- (e) State whether post-hoc analysis is necessary and explain why.

</div>

#### Step 1 : Hypotheses

Let $\mu_A, \mu_B, \mu_C, \mu_D, \mu_E$ = true mean accuracy scores for the five training methods.

$$H_0: \mu_A = \mu_B = \mu_C = \mu_D = \mu_E \quad \text{(all group means are equal)}$$
$$H_1: \text{at least one mean differs}$$

#### Step 2 : Why This Test

*We have five independent groups and want to test whether any training method produces a systematically different mean accuracy. One-way ANOVA is appropriate because it compares more than two group means simultaneously without inflating the Type I error rate. The F ratio quantifies how much between-group variance (explained by method choice) exceeds within-group variance (random noise). Independence, approximate normality, and homoscedasticity are assumed.*

#### Step 3 : Test Statistic

$$F = \frac{MS_{\text{between}}}{MS_{\text{within}}}, \qquad MS_{\text{between}} = \frac{SS_{\text{between}}}{k-1}, \qquad MS_{\text{within}} = \frac{SS_{\text{within}}}{N-k}$$

$$SS_{\text{between}} = \sum_{j} n_j (\bar{x}_j - \bar{x}_{\text{grand}})^2 \qquad SS_{\text{within}} = \sum_{j}\sum_{i} (x_{ij} - \bar{x}_j)^2$$

In [16]:
A = np.array([78, 82, 75, 80, 79, 81])
B = np.array([91, 88, 85, 90, 87, 89])
C = np.array([72, 70, 74, 73, 71, 69])
D = np.array([95, 92, 96, 94, 91, 93])
E = np.array([84, 83, 82, 81, 85, 86])

alpha  = 0.01
groups = [A, B, C, D, E]
k      = len(groups)
n      = len(A)
N      = k * n

In [17]:
mu_A = np.mean(A)
mu_B = np.mean(B)
mu_C = np.mean(C)
mu_D = np.mean(D)
mu_E = np.mean(E)
mu   = np.mean(np.concatenate(groups))

print(f'mu_A : {mu_A:.4f}')
print(f'mu_B : {mu_B:.4f}')
print(f'mu_C : {mu_C:.4f}')
print(f'mu_D : {mu_D:.4f}')
print(f'mu_E : {mu_E:.4f}')
print(f'mu   : {mu:.4f}')

mu_A : 79.1667
mu_B : 88.3333
mu_C : 71.5000
mu_D : 93.5000
mu_E : 83.5000
mu   : 83.2000


In [18]:
means     = np.array([mu_A, mu_B, mu_C, mu_D, mu_E])
ss_bw     = n * np.sum((means - mu)**2)
ss_within = sum(np.sum((g - g.mean())**2) for g in groups)
ss_total  = ss_bw + ss_within

print(f'ss_bw    : {ss_bw:.4f}')
print(f'ss_within: {ss_within:.4f}')
print(f'ss_total : {ss_total:.4f}')

ss_bw    : 1714.1333
ss_within: 106.6667
ss_total : 1820.8000


In [19]:
df_bw     = k - 1
df_within = N - k

ms_bw     = ss_bw / df_bw
ms_within = ss_within / df_within
f_stat    = ms_bw / ms_within

print(f'df_bw    : {df_bw}')
print(f'df_within: {df_within}')
print(f'ms_bw    : {ms_bw:.4f}')
print(f'ms_within: {ms_within:.4f}')
print(f'f_stat   : {f_stat:.4f}')

df_bw    : 4
df_within: 25
ms_bw    : 428.5333
ms_within: 4.2667
f_stat   : 100.4375


#### Step 4 : Critical Value Method

In [20]:
f_crit = f.ppf(1 - alpha, df_bw, df_within)

print(f'f_stat : {f_stat:.4f}')
print(f'f_crit : {f_crit:.4f}')

f_stat : 100.4375
f_crit : 4.1774


**F_stat (100.4375) >> F_crit (4.1774) → Reject H₀**

#### Step 5 : p-value Method

In [21]:
pval = f.sf(f_stat, df_bw, df_within)
print(f'p_val : {pval:.2e}')

p_val : 5.05e-15


**p_val (~5.05e-15) << 0.01 → Reject H₀**

#### Step 6 : Final Conclusion

*η² measures the proportion of total variance explained by the training method factor. Values above 0.14 are large by convention. Since H₀ is rejected and k = 5, we know at least one pair of means differs but not which pairs. Post-hoc testing (e.g., Tukey HSD or Bonferroni) is therefore necessary to identify the specific methods that differ from one another.*

$$\eta^2 = \frac{SS_{\text{between}}}{SS_{\text{total}}}$$

In [22]:
eta_sq = ss_bw / ss_total
print(f'eta_sq : {eta_sq:.4f}')

eta_sq : 0.9414


In [23]:
f_stat_, pval_ = f_oneway(A, B, C, D, E)
print(f'f_stat (scipy) : {f_stat_:.4f}')
print(f'p_val  (scipy) : {pval_:.2e}')

f_stat (scipy) : 100.4375
p_val  (scipy) : 5.05e-15


#### ✅ Final Conclusion

| Metric | Value |
|---|---|
| Test statistic (F) | 100.4375 |
| Critical value | 4.1774 |
| p-value | 5.05e-15 |
| df (between, within) | 4, 25 |
| α | 0.01 |
| η² | 0.9414 |
| Decision | **Reject H₀** |

There is overwhelming evidence that training method affects model accuracy. The F statistic (100.44) vastly exceeds the critical value (4.18) and the p-value is effectively zero. η² = 0.9414 means the choice of training method explains about 94.1% of the total variance in accuracy scores, an exceptionally large effect. Post-hoc analysis is needed to identify which specific method pairs differ.

<div style="background-color:#eef3fb; border-left:5px solid #4a7abf; padding:14px 18px; border-radius:6px; margin-bottom:8px;">

📌 **Question 4 : One-Way ANOVA (Training Program Reaction Times)**

A sports scientist compares reaction times (ms) under 5 training programs:

- P1: `[210, 215, 220, 205, 218]`
- P2: `[198, 202, 205, 200, 199]`
- P3: `[260, 230, 245, 240, 255]`
- P4: `[190, 188, 192, 191, 189]`
- P5: `[225, 228, 230, 226, 224]`

Perform a complete one-way ANOVA at α = 0.01.

- (a) Compute all ANOVA quantities manually.
- (b) Construct the complete ANOVA table.
- (c) Test significance using both critical-value and p-value approaches.
- (d) Compute η² and interpret the magnitude of the effect.
- (e) Explain how large within-group variability influences ANOVA results.

</div>

#### Step 1 : Hypotheses

Let $\mu_1, \mu_2, \mu_3, \mu_4, \mu_5$ = true mean reaction times for training programs P1 through P5.

$$H_0: \mu_1 = \mu_2 = \mu_3 = \mu_4 = \mu_5 \quad \text{(all group means are equal)}$$
$$H_1: \text{at least one mean differs}$$

#### Step 2 : Why This Test

*We have five independent groups and want to test whether training program affects mean reaction time. One-way ANOVA is appropriate because it compares more than two group means simultaneously without inflating the Type I error rate. The F ratio quantifies how much between-group variance (explained by program choice) exceeds within-group variance (random noise). Independence, approximate normality, and homoscedasticity are assumed.*

#### Step 3 : Test Statistic

$$F = \frac{MS_{\text{between}}}{MS_{\text{within}}}, \qquad MS_{\text{between}} = \frac{SS_{\text{between}}}{k-1}, \qquad MS_{\text{within}} = \frac{SS_{\text{within}}}{N-k}$$

$$SS_{\text{between}} = \sum_{j} n_j (\bar{x}_j - \bar{x}_{\text{grand}})^2 \qquad SS_{\text{within}} = \sum_{j}\sum_{i} (x_{ij} - \bar{x}_j)^2$$

In [24]:
p1 = np.array([210, 215, 220, 205, 218])
p2 = np.array([198, 202, 205, 200, 199])
p3 = np.array([260, 230, 245, 240, 255])
p4 = np.array([190, 188, 192, 191, 189])
p5 = np.array([225, 228, 230, 226, 224])

alpha  = 0.01
groups = [p1, p2, p3, p4, p5]
k      = len(groups)
n      = len(p1)
N      = k * n

In [25]:
mu_1 = np.mean(p1)
mu_2 = np.mean(p2)
mu_3 = np.mean(p3)
mu_4 = np.mean(p4)
mu_5 = np.mean(p5)
mu   = np.mean(np.concatenate(groups))

print(f'mu_1 : {mu_1:.4f}')
print(f'mu_2 : {mu_2:.4f}')
print(f'mu_3 : {mu_3:.4f}')
print(f'mu_4 : {mu_4:.4f}')
print(f'mu_5 : {mu_5:.4f}')
print(f'mu   : {mu:.4f}')

mu_1 : 213.6000
mu_2 : 200.8000
mu_3 : 246.0000
mu_4 : 190.0000
mu_5 : 226.6000
mu   : 215.4000


In [26]:
means     = np.array([mu_1, mu_2, mu_3, mu_4, mu_5])
ss_bw     = n * np.sum((means - mu)**2)
ss_within = sum(np.sum((g - g.mean())**2) for g in groups)
ss_total  = ss_bw + ss_within

print(f'ss_bw    : {ss_bw:.4f}')
print(f'ss_within: {ss_within:.4f}')
print(f'ss_total : {ss_total:.4f}')

ss_bw    : 9616.8000
ss_within: 783.2000
ss_total : 10400.0000


In [27]:
df_bw     = k - 1
df_within = N - k

ms_bw     = ss_bw / df_bw
ms_within = ss_within / df_within
f_stat    = ms_bw / ms_within

print(f'df_bw    : {df_bw}')
print(f'df_within: {df_within}')
print(f'ms_bw    : {ms_bw:.4f}')
print(f'ms_within: {ms_within:.4f}')
print(f'f_stat   : {f_stat:.4f}')

df_bw    : 4
df_within: 20
ms_bw    : 2404.2000
ms_within: 39.1600
f_stat   : 61.3943


#### Step 4 : Critical Value Method

In [28]:
f_crit = f.ppf(1 - alpha, df_bw, df_within)

print(f'f_stat : {f_stat:.4f}')
print(f'f_crit : {f_crit:.4f}')

f_stat : 61.3943
f_crit : 4.4307


**F_stat (61.3943) >> F_crit (4.4307) → Reject H₀**

#### Step 5 : p-value Method

In [29]:
pval = f.sf(f_stat, df_bw, df_within)
print(f'p_val : {pval:.2e}')

p_val : 6.01e-11


**p_val (6.01e-11) << 0.01 → Reject H₀**

#### Step 6 : Final Conclusion

*η² measures the proportion of total variance explained by the training program factor. Values above 0.14 are large by convention. Large within-group variability inflates MS_within, which shrinks the F ratio and makes it harder to detect real differences between groups. If within-group spread were much larger here, the F statistic could fall below the critical value even when group means differ substantially.*

$$\eta^2 = \frac{SS_{\text{between}}}{SS_{\text{total}}}$$

In [30]:
eta_sq = ss_bw / ss_total
print(f'eta_sq : {eta_sq:.4f}')

eta_sq : 0.9247


In [31]:
f_stat_, pval_ = f_oneway(p1, p2, p3, p4, p5)
print(f'f_stat (scipy) : {f_stat_:.4f}')
print(f'p_val  (scipy) : {pval_:.2e}')

f_stat (scipy) : 61.3943
p_val  (scipy) : 6.01e-11


#### ✅ Final Conclusion

| Metric | Value |
|---|---|
| Test statistic (F) | 61.3943 |
| Critical value | 4.4307 |
| p-value | 6.01e-11 |
| df (between, within) | 4, 20 |
| α | 0.01 |
| η² | 0.9247 |
| Decision | **Reject H₀** |

There is overwhelming evidence that training program affects reaction time. The F statistic (61.39) far exceeds the critical value (4.43) and the p-value is effectively zero. η² = 0.9247 means the choice of training program explains about 92.5% of the total variance in reaction times, an exceptionally large effect. Post-hoc analysis is needed to identify which specific program pairs differ significantly.

<div style="background-color:#eef3fb; border-left:5px solid #4a7abf; padding:14px 18px; border-radius:6px; margin-bottom:8px;">

📌 **Question 5 : One-Way ANOVA with Outlier Analysis (Therapy Stress Scores)**

A psychologist studies stress scores under 4 therapy methods:

- T1: `[40, 42, 41, 39, 43, 40]`
- T2: `[55, 57, 58, 54, 56, 55]`
- T3: `[48, 47, 49, 50, 46, 48]`
- T4: `[35, 34, 36, 33, 37, 90]`

Perform one-way ANOVA at α = 0.05.

- (a) Compute all ANOVA quantities and complete the ANOVA table.
- (b) Test whether therapy means differ significantly.
- (c) Compute η² and interpret the treatment effect size.
- (d) Explain how the outlier in T4 influences SSW, MSW, and the F-statistic.
- (e) State whether ANOVA assumptions may be violated and justify your answer.

</div>

#### Step 1 : Hypotheses

Let $\mu_1, \mu_2, \mu_3, \mu_4$ = true mean stress scores for therapy methods T1 through T4.

$$H_0: \mu_1 = \mu_2 = \mu_3 = \mu_4 \quad \text{(all group means are equal)}$$
$$H_1: \text{at least one mean differs}$$

#### Step 2 : Why This Test

*We have four independent groups and want to test whether therapy method affects mean stress score. One-way ANOVA is appropriate because it compares more than two group means simultaneously without inflating the Type I error rate. The F ratio quantifies how much between-group variance (explained by therapy choice) exceeds within-group variance (random noise). Independence, approximate normality, and homoscedasticity are assumed. Notably, T4 contains a suspected outlier (90) that may violate these assumptions.*

#### Step 3 : Test Statistic

$$F = \frac{MS_{\text{between}}}{MS_{\text{within}}}, \qquad MS_{\text{between}} = \frac{SS_{\text{between}}}{k-1}, \qquad MS_{\text{within}} = \frac{SS_{\text{within}}}{N-k}$$

$$SS_{\text{between}} = \sum_{j} n_j (\bar{x}_j - \bar{x}_{\text{grand}})^2 \qquad SS_{\text{within}} = \sum_{j}\sum_{i} (x_{ij} - \bar{x}_j)^2$$

In [32]:
T1 = np.array([40, 42, 41, 39, 43, 40])
T2 = np.array([55, 57, 58, 54, 56, 55])
T3 = np.array([48, 47, 49, 50, 46, 48])
T4 = np.array([35, 34, 36, 33, 37, 90])

alpha  = 0.05
groups = [T1, T2, T3, T4]
k      = len(groups)
n      = len(T1)
N      = k * n

In [33]:
mu_T1 = np.mean(T1)
mu_T2 = np.mean(T2)
mu_T3 = np.mean(T3)
mu_T4 = np.mean(T4)
mu    = np.mean(np.concatenate(groups))

print(f'mu_T1 : {mu_T1:.4f}')
print(f'mu_T2 : {mu_T2:.4f}')
print(f'mu_T3 : {mu_T3:.4f}')
print(f'mu_T4 : {mu_T4:.4f}')
print(f'mu    : {mu:.4f}')

mu_T1 : 40.8333
mu_T2 : 55.8333
mu_T3 : 48.0000
mu_T4 : 44.1667
mu    : 47.2083


In [34]:
means     = np.array([mu_T1, mu_T2, mu_T3, mu_T4])
ss_bw     = n * np.sum((means - mu)**2)
ss_within = sum(np.sum((g - g.mean())**2) for g in groups)
ss_total  = ss_bw + ss_within

print(f'ss_bw    : {ss_bw:.4f}')
print(f'ss_within: {ss_within:.4f}')
print(f'ss_total : {ss_total:.4f}')

ss_bw    : 749.4583
ss_within: 2562.5000
ss_total : 3311.9583


In [35]:
df_bw     = k - 1
df_within = N - k

ms_bw     = ss_bw / df_bw
ms_within = ss_within / df_within
f_stat    = ms_bw / ms_within

print(f'df_bw    : {df_bw}')
print(f'df_within: {df_within}')
print(f'ms_bw    : {ms_bw:.4f}')
print(f'ms_within: {ms_within:.4f}')
print(f'f_stat   : {f_stat:.4f}')

df_bw    : 3
df_within: 20
ms_bw    : 249.8194
ms_within: 128.1250
f_stat   : 1.9498


#### Step 4 : Critical Value Method

In [36]:
f_crit = f.ppf(1 - alpha, df_bw, df_within)

print(f'f_stat : {f_stat:.4f}')
print(f'f_crit : {f_crit:.4f}')

f_stat : 1.9498
f_crit : 3.0984


**F_stat (1.9498) < F_crit (3.0984) → Fail to Reject H₀**

#### Step 5 : p-value Method

In [37]:
pval = f.sf(f_stat, df_bw, df_within)
print(f'p_val : {pval:.6f}')

p_val : 0.154111


**p_val (0.154111) > 0.05 → Fail to Reject H₀**

#### Step 6 : Final Conclusion

*η² measures the proportion of total variance explained by the therapy factor. However, the outlier in T4 (score = 90 vs. group values of 33 to 37) inflates SS_within massively: T4's within-group SS with the outlier is 2530.83, compared to just 10.00 without it. This inflated MS_within shrinks the F ratio and reduces the test's ability to detect real between-group differences. The ANOVA assumptions of homoscedasticity and normality are very likely violated because T4's variance is orders of magnitude larger than the other groups, making the result unreliable.*

$$\eta^2 = \frac{SS_{\text{between}}}{SS_{\text{total}}}$$

In [38]:
eta_sq = ss_bw / ss_total
print(f'eta_sq : {eta_sq:.4f}')

eta_sq : 0.2263


In [39]:
f_stat_, pval_ = f_oneway(T1, T2, T3, T4)
print(f'f_stat (scipy) : {f_stat_:.4f}')
print(f'p_val  (scipy) : {pval_:.6f}')

f_stat (scipy) : 1.9498
p_val  (scipy) : 0.154111


#### ✅ Final Conclusion

| Metric | Value |
|---|---|
| Test statistic (F) | 1.9498 |
| Critical value | 3.0984 |
| p-value | 0.154111 |
| df (between, within) | 3, 20 |
| α | 0.05 |
| η² | 0.2263 |
| Decision | **Fail to Reject H₀** |

There is insufficient evidence to conclude that therapy method affects stress scores at α = 0.05. The F statistic (1.95) falls below the critical value (3.10) and the p-value (0.154) exceeds the threshold. However, this result is heavily distorted by the outlier in T4 (value = 90). Without the outlier, T4's within-group SS drops from 2530.83 to 10.00, meaning the outlier is responsible for the vast majority of SS_within (2562.50 total). This inflated MS_within suppresses the F ratio and likely masks a real treatment effect. η² = 0.2263 suggests the therapy factor nominally explains about 22.6% of variance, but this estimate is untrustworthy given the assumption violations. The outlier in T4 violates both normality and homoscedasticity, making ANOVA inappropriate without first addressing the outlier through removal, transformation, or a robust non-parametric alternative such as the Kruskal-Wallis test.

<div style="background-color:#eef3fb; border-left:5px solid #4a7abf; padding:14px 18px; border-radius:6px; margin-bottom:8px;">

📌 **Question 6 : High-Level ANOVA Interpretation (Diet Weight Loss)**

Five diets are tested for average weight loss (kg). The observations are:

- Diet A: `[4, 5, 6, 5, 4, 5]`
- Diet B: `[8, 9, 7, 10, 8, 9]`
- Diet C: `[3, 2, 4, 3, 2, 3]`
- Diet D: `[11, 12, 10, 13, 11, 12]`
- Diet E: `[6, 7, 5, 6, 7, 6]`

Perform a complete one-way ANOVA at α = 0.01.

- (a) Compute SSB, SSW, SST, degrees of freedom, mean squares, and F-statistic.
- (b) Test the hypothesis using the critical value method.
- (c) Compute the p-value and interpret it.
- (d) Compute η² and classify the effect size using Cohen's guidelines.
- (e) Explain why rejecting H₀ does NOT imply every pair of means differs significantly.

</div>

#### Step 1 : Hypotheses

Let $\mu_A, \mu_B, \mu_C, \mu_D, \mu_E$ = true mean weight loss (kg) for diets A through E.

$$H_0: \mu_A = \mu_B = \mu_C = \mu_D = \mu_E \quad \text{(all group means are equal)}$$
$$H_1: \text{at least one mean differs}$$

#### Step 2 : Why This Test

*We have five independent groups and want to test whether diet choice affects mean weight loss. One-way ANOVA is appropriate because it compares more than two group means simultaneously without inflating the Type I error rate. The F ratio quantifies how much between-group variance (explained by diet choice) exceeds within-group variance (random noise). Independence, approximate normality, and homoscedasticity are assumed.*

#### Step 3 : Test Statistic

$$F = \frac{MS_{\text{between}}}{MS_{\text{within}}}, \qquad MS_{\text{between}} = \frac{SS_{\text{between}}}{k-1}, \qquad MS_{\text{within}} = \frac{SS_{\text{within}}}{N-k}$$

$$SS_{\text{between}} = \sum_{j} n_j (\bar{x}_j - \bar{x}_{\text{grand}})^2 \qquad SS_{\text{within}} = \sum_{j}\sum_{i} (x_{ij} - \bar{x}_j)^2$$

In [40]:
# --- Diet weight loss data (kg) ---
A = np.array([4, 5, 6, 5, 4, 5])
B = np.array([8, 9, 7, 10, 8, 9])
C = np.array([3, 2, 4, 3, 2, 3])
D = np.array([11, 12, 10, 13, 11, 12])
E = np.array([6, 7, 5, 6, 7, 6])

alpha  = 0.01
groups = [A, B, C, D, E]
k      = len(groups)   # number of groups
n      = len(A)        # observations per group
N      = n * k         # total observations

In [41]:
# --- Group means and grand mean ---
mu_A = np.mean(A)
mu_B = np.mean(B)
mu_C = np.mean(C)
mu_D = np.mean(D)
mu_E = np.mean(E)
grand_mean = np.mean(np.concatenate(groups))

print(f'mu_A       : {mu_A:.4f}')
print(f'mu_B       : {mu_B:.4f}')
print(f'mu_C       : {mu_C:.4f}')
print(f'mu_D       : {mu_D:.4f}')
print(f'mu_E       : {mu_E:.4f}')
print(f'grand_mean : {grand_mean:.4f}')

mu_A       : 4.8333
mu_B       : 8.5000
mu_C       : 2.8333
mu_D       : 11.5000
mu_E       : 6.1667
grand_mean : 6.7667


In [42]:
# --- Sum of Squares ---
means     = np.array([mu_A, mu_B, mu_C, mu_D, mu_E])
SS_bw     = n * np.sum((means - grand_mean)**2)          # between-group SS
SS_within = sum(np.sum((g - g.mean())**2) for g in groups)  # within-group SS
SS_total  = SS_bw + SS_within

print(f'SS_bw    : {SS_bw:.4f}')
print(f'SS_within: {SS_within:.4f}')
print(f'SS_total : {SS_total:.4f}')

SS_bw    : 269.8667
SS_within: 19.5000
SS_total : 289.3667


In [43]:
# --- Degrees of freedom, Mean Squares, F-statistic ---
df_bw     = k - 1       # between-group df
df_within = N - k       # within-group df

MS_bw     = SS_bw / df_bw
MS_within = SS_within / df_within
F_stat    = MS_bw / MS_within

print(f'df_bw    : {df_bw}')
print(f'df_within: {df_within}')
print(f'MS_bw    : {MS_bw:.4f}')
print(f'MS_within: {MS_within:.4f}')
print(f'F_stat   : {F_stat:.4f}')

df_bw    : 4
df_within: 25
MS_bw    : 67.4667
MS_within: 0.7800
F_stat   : 86.4957


#### Step 4 : Critical Value Method

In [44]:
# --- Critical value at alpha = 0.01 ---
F_crit = f.ppf(1 - alpha, df_bw, df_within)

print(f'F_stat : {F_stat:.4f}')
print(f'F_crit : {F_crit:.4f}')

F_stat : 86.4957
F_crit : 4.1774


**F_stat (86.4957) >> F_crit (4.1774) → Reject H₀**

#### Step 5 : p-value Method

In [45]:
# --- p-value from the upper tail of the F distribution ---
pval = f.sf(F_stat, df_bw, df_within)
print(f'p_val : {pval:.2e}')

p_val : 2.88e-14


**p_val (2.88e-14) << 0.01 → Reject H₀**

#### Step 6 : Final Conclusion

*η² measures the proportion of total variance explained by the diet factor. By Cohen\'s guidelines, η² > 0.14 is a large effect. Rejecting H₀ only tells us that at least one pair of group means differs; it does NOT identify which pairs differ. With k = 5 groups there are C(5,2) = 10 possible pairwise comparisons, and ANOVA cannot distinguish among them. Post-hoc tests (e.g., Tukey HSD or Bonferroni) are required to localise which specific diets produce significantly different weight loss.*

$$\eta^2 = \frac{SS_{\text{between}}}{SS_{\text{total}}}$$

In [46]:
# --- Effect size: eta-squared ---
eta_sq = SS_bw / SS_total
print(f'eta_sq : {eta_sq:.4f}')

eta_sq : 0.9326


In [47]:
# --- Verification via scipy ---
F_stat2, pval2 = f_oneway(A, B, C, D, E)
print(f'F_stat (scipy) : {F_stat2:.4f}')
print(f'p_val  (scipy) : {pval2:.2e}')

F_stat (scipy) : 86.4957
p_val  (scipy) : 2.88e-14


#### ✅ Final Conclusion

| Metric | Value |
|---|---|
| Test statistic (F) | 86.4957 |
| Critical value | 4.1774 |
| p-value | 2.88e-14 |
| df (between, within) | 4, 25 |
| α | 0.01 |
| η² | 0.9326 |
| Decision | **Reject H₀** |

There is overwhelming evidence that diet choice affects weight loss. The F statistic (86.50) far exceeds the critical value (4.18) and the p-value (2.88e-14) is effectively zero, so we reject H₀ at α = 0.01. η² = 0.9326 means diet choice explains about 93.3% of the total variance in weight loss, which is an exceptionally large effect by Cohen's guidelines (large: η² > 0.14). However, rejecting H₀ only establishes that the five diet means are not all equal; it does not reveal which specific diets differ from one another. Post-hoc analysis (e.g., Tukey HSD) is therefore necessary to identify the specific pairs of diets with significantly different mean weight loss.

<div style="background-color:#eef3fb; border-left:5px solid #4a7abf; padding:14px 18px; border-radius:6px; margin-bottom:8px;">

📌 **Question 7 : One-Way ANOVA (Diet Programs — Weight Loss)**

A nutritionist compares 4 diet programs based on weight loss (kg) after 8 weeks:

- **Diet A:** `[5, 6, 4, 7, 5, 6]`
- **Diet B:** `[8, 9, 7, 10, 8, 9]`
- **Diet C:** `[3, 2, 4, 3, 5, 2]`
- **Diet D:** `[11, 12, 10, 13, 11, 12]`

Perform a complete one-way ANOVA at α = 0.05.

- (a) Compute group means, grand mean, SSB, SSW, SST.
- (b) Construct the ANOVA table.
- (c) Test significance using F-critical and p-value methods.
- (d) Compute η² and interpret the effect size.

</div>

#### Step 1 : Hypotheses

Let $\mu_A, \mu_B, \mu_C, \mu_D$ = true mean weight loss (kg) for diet programs A through D.

$$H_0: \mu_A = \mu_B = \mu_C = \mu_D \quad \text{(all group means are equal)}$$
$$H_1: \text{at least one mean differs}$$

#### Step 2 : Why This Test

*We have four independent groups and want to test whether diet program affects mean weight loss. One-way ANOVA is appropriate because it compares more than two group means simultaneously without inflating the Type I error rate. The F ratio quantifies how much between-group variance (explained by diet choice) exceeds within-group variance (random noise). Independence, approximate normality, and homoscedasticity are assumed.*

#### Step 3 : Test Statistic

$$F = \frac{MS_{\text{between}}}{MS_{\text{within}}}, \qquad MS_{\text{between}} = \frac{SS_{\text{between}}}{k-1}, \qquad MS_{\text{within}} = \frac{SS_{\text{within}}}{N-k}$$

$$SS_{\text{between}} = \sum_{j} n_j (\bar{x}_j - \bar{x}_{\text{grand}})^2 \qquad SS_{\text{within}} = \sum_{j}\sum_{i} (x_{ij} - \bar{x}_j)^2$$

In [48]:
# weight loss (kg) recorded for each subject under their assigned diet
A = np.array([5, 6, 4, 7, 5, 6])
B = np.array([8, 9, 7, 10, 8, 9])
C = np.array([3, 2, 4, 3, 5, 2])
D = np.array([11, 12, 10, 13, 11, 12])

In [49]:
alpha = 0.05
groups = [A,B,C,D]
k = len(groups)   # number of diet groups
n = len(A)        # observations per group
N = k * n         # total observations

In [50]:
mu_A = np.mean(A)
mu_B = np.mean(B)
mu_C = np.mean(C)
mu_D = np.mean(D)
mu = np.mean(np.concatenate([A,B,C,D]))  # grand mean across all groups
mu

np.float64(7.166666666666667)

In [51]:
means = [mu_A,  mu_B, mu_C, mu_D]

In [52]:
# SS_between: how much group means deviate from the grand mean
SS_bw = n * np.sum((means-mu)**2)
SS_bw

np.float64(235.99999999999997)

In [53]:
# SS_within: variability of observations around their own group mean
SS_within = sum(np.sum((g - g.mean())**2) for g in groups)
SS_within

np.float64(23.333333333333332)

In [54]:
SS_total = SS_bw + SS_within

In [55]:
df_bw = k-1      # between-group df
df_within = N-k  # within-group df

In [56]:
# MS = SS / df; converts sums of squares into average variance per df
MS_bw = SS_bw/df_bw
MS_within = SS_within/df_within

In [57]:
# F ratio: signal (between-group variance) relative to noise (within-group variance)
F_stat = MS_bw/MS_within
F_stat

np.float64(67.42857142857143)

#### Step 4 : Critical Value Method

In [58]:
F_crit = f.ppf(1-alpha, df_bw, df_within)
F_crit

np.float64(3.098391212140781)

**F_stat >> F_crit → Reject H₀**

#### Step 5 : p-value Method

In [59]:
pval = f.sf(F_stat, df_bw, df_within)
pval

np.float64(1.232731988544722e-10)

**p_val < 0.05 → Reject H₀**

#### Step 6 : Effect Size & Final Conclusion

*η² (eta-squared) measures the proportion of total variance explained by the diet factor. By Cohen's guidelines, η² > 0.14 is a large effect.*

$$\eta^2 = \frac{SS_{\text{between}}}{SS_{\text{total}}}$$

In [60]:
# eta-squared: proportion of total variance explained by the diet factor
eta_sq = SS_bw/SS_total
eta_sq

np.float64(0.910025706940874)

#### ✅ Final Conclusion

| Metric | Value |
|---|---|
| Test statistic (F) | 67.4286 |
| Critical value | 3.0984 |
| p-value | 1.23e-10 |
| df (between, within) | 3, 20 |
| α | 0.05 |
| η² | 0.9100 |
| Decision | **Reject H₀** |

At α = 0.05, there is strong evidence that diet program significantly affects weight loss. The F statistic (67.43) far exceeds the critical value (3.10) and the p-value (1.23e-10) is effectively zero, so we reject H₀. η² = 0.9100 means diet choice explains about 91% of the total variance in weight loss — an exceptionally large effect by Cohen's guidelines (large: η² > 0.14). Post-hoc testing (e.g., Tukey HSD) would be needed to identify which specific diet pairs differ significantly.

<div style="background-color:#eef3fb; border-left:5px solid #4a7abf; padding:14px 18px; border-radius:6px; margin-bottom:8px;">

📌 **Question 8 : One-Way ANOVA with Unequal Sample Sizes**

Three gyms are compared on monthly member attendance. Groups have **different** sizes:

- Gym A (n=4): `[120, 135, 128, 141]`
- Gym B (n=6): `[98, 102, 95, 110, 99, 106]`
- Gym C (n=5): `[155, 162, 149, 158, 161]`

Perform one-way ANOVA at α = 0.05 with **unequal** group sizes. Adapt the SS_between formula accordingly. Compute η² and state your conclusion.

</div>

#### Step 1 : Hypotheses

Let $\mu_A, \mu_B, \mu_C$ = true mean monthly attendance for Gyms A, B, and C.

$$H_0: \mu_A = \mu_B = \mu_C \quad \text{(all gym means are equal)}$$
$$H_1: \text{at least one mean differs}$$

#### Step 2 : Why This Test

*Three independent groups with unequal sample sizes — one-way ANOVA is still appropriate. When group sizes differ, the SS_between formula must weight each group's squared deviation by its own $n_j$ rather than a common $n$. The F ratio structure, assumptions (independence, normality, homoscedasticity), and decision rules are otherwise identical to the balanced case.*

#### Step 3 : Test Statistic

$$F = \frac{MS_{\text{between}}}{MS_{\text{within}}}, \qquad MS_{\text{between}} = \frac{SS_{\text{between}}}{k-1}, \qquad MS_{\text{within}} = \frac{SS_{\text{within}}}{N-k}$$

$$SS_{\text{between}} = \sum_{j} n_j (\bar{x}_j - \bar{x}_{\text{grand}})^2 \qquad SS_{\text{within}} = \sum_{j}\sum_{i} (x_{ij} - \bar{x}_j)^2$$

In [61]:
# monthly attendance observations for each gym
A = np.array([120, 135, 128, 141])
B = np.array([98, 102, 95, 110, 99, 106])
C = np.array([155, 162, 149, 158, 161])

In [62]:
# test parameters: significance level, group list, group count, per-group sizes, total N
alpha = 0.05
groups = [A, B, C]
k = len(groups)
ns = np.array([len(g) for g in groups])
N = np.sum(ns)

In [63]:
# group means and grand mean (weighted equally across all observations)
means = np.array([g.mean() for g in groups])
grand_mean = np.mean(np.concatenate(groups))
grand_mean

np.float64(127.93333333333334)

In [64]:
# SS_between: each group's squared deviation from the grand mean, weighted by its own n_j
SS_bw = sum(ns * (means - grand_mean)**2)
SS_bw

np.float64(8401.599999999999)

In [65]:
# SS_within: sum of squared deviations of each observation from its group mean
SS_within = sum(sum((g - g.mean())**2) for g in groups)
SS_within

np.float64(509.33333333333337)

In [66]:
# SS_total: total variability in the data
SS_total = SS_bw + SS_within
SS_total

np.float64(8910.933333333332)

In [67]:
df_bw = k - 1
df_within = N - k

In [68]:
# MS = SS / df; converts sums of squares into average variance per degree of freedom
MS_bw = SS_bw/df_bw
MS_within = SS_within/df_within

In [69]:
# F ratio: between-group variance (signal) relative to within-group variance (noise)
F_stat = MS_bw/MS_within
F_stat

np.float64(98.97172774869107)

#### Step 4 : Critical Value Method

In [70]:
# critical F value at alpha for df_bw and df_within
F_crit = f.ppf(1-alpha, df_bw, df_within)
F_crit

np.float64(3.8852938346523924)

**F_stat (98.9717) > F_crit (3.8853) → Reject H₀**

#### Step 5 : p-value Method

In [71]:
# p-value: probability of observing F_stat or larger under H₀
pval = f.sf(F_stat, df_bw, df_within)
pval

np.float64(3.487172490258324e-08)

**p_val (3.49e-08) < 0.05 → Reject H₀**

#### Step 6 : Effect Size & Final Conclusion

*η² (eta-squared) measures the proportion of total variance explained by the gym factor. By Cohen's guidelines, η² > 0.14 is a large effect.*

$$\eta^2 = \frac{SS_{\text{between}}}{SS_{\text{total}}}$$

In [72]:
# eta-squared: proportion of total variance explained by gym membership
eta_sq = SS_bw/SS_total
eta_sq

np.float64(0.9428417524539142)

#### ✅ Final Conclusion

| Metric | Value |
|---|---|
| Test statistic (F) | 98.9717 |
| Critical value | 3.8853 |
| p-value | 3.49e-08 |
| df (between, within) | 2, 12 |
| α | 0.05 |
| η² | 0.9428 |
| Decision | **Reject H₀** |

At α = 0.05, there is overwhelming evidence that gym membership significantly affects monthly attendance. The F statistic (98.97) far exceeds the critical value (3.89) and the p-value (3.49e-08) is effectively zero, so we reject H₀. η² = 0.9428 means gym choice explains approximately 94.3% of the total variance in attendance — an exceptionally large effect by Cohen's guidelines (large: η² > 0.14). Gym C consistently records the highest attendance while Gym B records the lowest.

<div style="background-color:#eef3fb; border-left:5px solid #4a7abf; padding:14px 18px; border-radius:6px; margin-bottom:8px;">

📌 **Question 9 : One-Way ANOVA + Tukey HSD Post-Hoc**

A pharmaceutical company tests four dosage levels of a drug on blood pressure reduction (mmHg):

- Placebo (n=5): `[2, 1, 3, 2, 1]`
- Low Dose (n=5): `[8, 7, 9, 8, 6]`
- Mid Dose (n=5): `[14, 15, 13, 16, 14]`
- High Dose (n=5): `[20, 22, 19, 21, 18]`

Perform a one-way ANOVA at α = 0.05 to determine whether mean blood pressure reduction differs among dosage levels.

- (a) Compute all ANOVA quantities and construct the ANOVA table.
- (b) Test significance using both critical value and p-value approaches.
- (c) If the result is significant, run Tukey HSD post-hoc to identify which dosage pairs differ.
- (d) Compute η² and interpret the effect size.
- (e) Explain what Tukey HSD controls for that individual t-tests do not.

</div>

#### Step 1 : Hypotheses

Let $\mu_P, \mu_L, \mu_M, \mu_H$ = true mean blood pressure reduction (mmHg) for Placebo, Low, Mid, and High dosage groups.

$$H_0: \mu_P = \mu_L = \mu_M = \mu_H \quad \text{(all dosage means are equal)}$$
$$H_1: \text{at least one mean differs}$$

#### Step 2 : Why This Test

*We have 4 independent groups and want to test whether drug dosage lowers blood pressure. One-way ANOVA is appropriate because it compares more than two group means simultaneously without inflating the Type I error rate. The F ratio quantifies how much between-group variance (explained by diet choice) exceeds within-group variance (random noise). Independence, approximate normality, and homoscedasticity are assumed.*

#### Step 3 : Test Statistic

$$F = \frac{MS_{\text{between}}}{MS_{\text{within}}}, \qquad MS_{\text{between}} = \frac{SS_{\text{between}}}{k-1}, \qquad MS_{\text{within}} = \frac{SS_{\text{within}}}{N-k}$$

$$SS_{\text{between}} = \sum_{j} n_j (\bar{x}_j - \bar{x}_{\text{grand}})^2 \qquad SS_{\text{within}} = \sum_{j}\sum_{i} (x_{ij} - \bar{x}_j)^2$$

In [73]:
P = np.array([2, 1, 3, 2, 1])
L = np.array([8, 7, 9, 8, 6])
M = np.array([14, 15, 13, 16, 14])
H = np.array([20, 22, 19, 21, 18])

In [74]:
alpha = 0.05
groups = np.array([P, L, M, H])
k = len(groups)
n = len(P)
N = k * n

In [75]:
means = [g.mean() for g in groups]
grand_mean = np.mean(np.concat(groups))
means, grand_mean

([np.float64(1.8), np.float64(7.6), np.float64(14.4), np.float64(20.0)],
 np.float64(10.95))

In [76]:
# SS_between: how much group means deviate from the grand mean
SS_bw = n* np.sum((means - grand_mean)**2)
SS_bw

np.float64(943.75)

In [77]:
# SS_within: variability of observations around their own group mean
SS_within = sum(sum((g- g.mean())**2) for g in groups)
SS_within

np.float64(23.2)

In [78]:
SS_total = SS_bw + SS_within
SS_total

np.float64(966.95)

In [79]:
df_bw = k - 1
df_within = N - k

In [80]:
# MS = SS / df; converts sums of squares into average variance per df
MS_bw = SS_bw/df_bw
MS_within = SS_within/df_within

In [81]:
# F ratio: signal (between-group variance) relative to noise (within-group variance)
F_stat = MS_bw/MS_within
F_stat

np.float64(216.95402298850573)

#### Step 4 : Critical Value Method

In [82]:
F_crit = f.ppf(1-alpha, df_bw, df_within)
F_crit

np.float64(3.2388715174535863)

**F_stat >> F_crit → Reject H₀**

#### Step 5 : p-value Method

In [83]:
pval = f.sf(F_stat, df_bw, df_within)
pval

np.float64(3.62691801323238e-13)

**p_val < 0.05 → Reject H₀**

In [84]:
F_stat, pval = f_oneway(P,L,M,H)
F_stat, pval

(np.float64(216.95402298850638), np.float64(3.626918013232301e-13))

In [85]:
result = tukey_hsd(*groups)
print(result)

Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)     -5.800     0.000    -7.979    -3.621
 (0 - 2)    -12.600     0.000   -14.779   -10.421
 (0 - 3)    -18.200     0.000   -20.379   -16.021
 (1 - 0)      5.800     0.000     3.621     7.979
 (1 - 2)     -6.800     0.000    -8.979    -4.621
 (1 - 3)    -12.400     0.000   -14.579   -10.221
 (2 - 0)     12.600     0.000    10.421    14.779
 (2 - 1)      6.800     0.000     4.621     8.979
 (2 - 3)     -5.600     0.000    -7.779    -3.421
 (3 - 0)     18.200     0.000    16.021    20.379
 (3 - 1)     12.400     0.000    10.221    14.579
 (3 - 2)      5.600     0.000     3.421     7.779



*Every dosage level differs significantly from every other dosage level. Increasing dosage produces statistically different blood pressure reductions at each level.*

#### Step 6 : Effect Size & Final Conclusion

*η² (eta-squared) measures the proportion of total variance explained by the dosage factor. By Cohen's guidelines, η² > 0.14 is a large effect. Since ANOVA only establishes that at least one pair differs, Tukey HSD is applied post-hoc: it compares every possible pair of group means while controlling the family-wise error rate (FWER) at α = 0.05. Running individual t-tests for each pair would inflate the Type I error rate from α to 1 − (1 − α)^C where C is the number of comparisons.*

$$\eta^2 = \frac{SS_{\text{between}}}{SS_{\text{total}}}$$

In [86]:
eta_sq = SS_bw/SS_total
eta_sq

np.float64(0.9760070324215315)

#### ✅ Final Conclusion

| Metric | Value |
|---|---|
| Test statistic (F) | 216.9540 |
| Critical value | 3.2389 |
| p-value | 3.63e-13 |
| SS Between | 943.7500 |
| SS Within | 23.2000 |
| SS Total | 966.9500 |
| MS Between | 314.5833 |
| MS Within | 1.4500 |
| df (between, within) | 3, 16 |
| α | 0.05 |
| η² | 0.9760 |
| Decision | **Reject H₀** |

Mean blood pressure reduction differs significantly across dosage levels. The F statistic (216.95) vastly exceeds the critical value (3.24) and the p-value (3.63e-13) is effectively zero, so we reject H₀ at α = 0.05. η² = 0.9760 means dosage level explains about 97.6% of the total variance in blood pressure reduction, an exceptionally large effect by Cohen's guidelines (large: η² > 0.14). Tukey HSD confirms that every dosage pair differs significantly: each step up in dosage produces a statistically distinct increase in blood pressure reduction. This demonstrates why post-hoc testing is essential after a significant ANOVA — the omnibus F-test only tells us that the means are not all equal; Tukey HSD pinpoints exactly which pairs drive that conclusion while keeping the family-wise error rate at 5%.

<div style="background-color:#eef3fb; border-left:5px solid #4a7abf; padding:14px 18px; border-radius:6px; margin-bottom:8px;">

📌 **Question 10 : One-Way ANOVA (Sleep Duration by Age Group)**

A sleep researcher records average nightly sleep (hours) for four age groups:

- Teens: `[7.5, 8.0, 7.8, 8.2, 7.9]`
- Adults: `[7.0, 6.8, 7.2, 6.9, 7.1]`
- Middle-aged: `[6.5, 6.3, 6.7, 6.4, 6.6]`
- Seniors: `[6.0, 5.8, 6.2, 5.9, 6.1]`

Perform a complete one-way ANOVA at α = 0.05.

- (a) Compute group means, grand mean, SSB, SSW, SST.
- (b) Construct the complete ANOVA table.
- (c) Test significance using both the critical value and p-value approaches.
- (d) Compute η² and interpret the effect size.
- (e) State whether age group significantly affects sleep duration and discuss any practical implications.

</div>

#### Step 1 : Hypotheses

Let $\mu_T, \mu_A, \mu_M, \mu_S$ = true mean nightly sleep (hours) for Teens, Adults, Middle-aged, and Seniors.

$$H_0: \mu_T = \mu_A = \mu_M = \mu_S \quad \text{(all age group means are equal)}$$
$$H_1: \text{at least one mean differs}$$

#### Step 2 : Why This Test

*We have four independent groups and want to test whether age affects average nightly sleep duration. One-way ANOVA is appropriate because it compares more than two group means simultaneously without inflating the Type I error rate. The F ratio quantifies how much between-group variance (explained by age group) exceeds within-group variance (random noise within each age group). Independence, approximate normality, and homoscedasticity are assumed.*

#### Step 3 : Test Statistic

$$F = \frac{MS_{\text{between}}}{MS_{\text{within}}}, \qquad MS_{\text{between}} = \frac{SS_{\text{between}}}{k-1}, \qquad MS_{\text{within}} = \frac{SS_{\text{within}}}{N-k}$$

$$SS_{\text{between}} = \sum_{j} n_j (\bar{x}_j - \bar{x}_{\text{grand}})^2 \qquad SS_{\text{within}} = \sum_{j}\sum_{i} (x_{ij} - \bar{x}_j)^2$$

In [87]:
# average nightly sleep (hours) for each subject by age group
T = np.array([7.5, 8.0, 7.8, 8.2, 7.9])
A = np.array([7.0, 6.8, 7.2, 6.9, 7.1])
M = np.array([6.5, 6.3, 6.7, 6.4, 6.6])
S = np.array([6.0, 5.8, 6.2, 5.9, 6.1])

In [88]:
alpha  = 0.05
groups = [T, A, M, S]
k      = len(groups)   # number of age groups
n      = len(T)        # observations per group
N      = k * n         # total observations

In [89]:
# group means and grand mean
means      = [g.mean() for g in groups]
grand_mean = np.mean(np.concatenate(groups))
means, grand_mean

([np.float64(7.88), np.float64(7.0), np.float64(6.5), np.float64(6.0)],
 np.float64(6.844999999999999))

In [90]:
# SS_between: how much group means deviate from the grand mean
SS_bw = n * np.sum((np.array(means) - grand_mean)**2)
SS_bw

np.float64(9.641499999999999)

In [91]:
# SS_within: variability of observations around their own group mean
SS_within = sum(np.sum((g - g.mean())**2) for g in groups)
SS_within

np.float64(0.5679999999999995)

In [92]:
SS_total = SS_bw + SS_within
SS_total

np.float64(10.209499999999998)

In [93]:
df_bw     = k - 1      # between-group df
df_within = N - k      # within-group df

In [94]:
# MS = SS / df; converts sums of squares into average variance per df
MS_bw     = SS_bw / df_bw
MS_within = SS_within / df_within

In [95]:
# F ratio: signal (between-group variance) relative to noise (within-group variance)
F_stat = MS_bw / MS_within
F_stat

np.float64(90.53051643192495)

#### Step 4 : Critical Value Method

In [96]:
F_crit = f.ppf(1-alpha, df_bw, df_within)
F_crit

np.float64(3.2388715174535863)

**F_stat >> F_crit → Reject H₀**

#### Step 5 : p-value Method

In [97]:
pval = f.sf(F_stat, df_bw, df_within)
pval

np.float64(2.987331635366862e-10)

**p_val << 0.05 → Reject H₀**

#### Step 6 : Effect Size & Final Conclusion

*η² (eta-squared) measures the proportion of total variance explained by the age group factor. By Cohen's guidelines, η² > 0.14 is a large effect. A significant result here suggests that natural physiological and lifestyle changes associated with aging produce systematically different sleep patterns across age groups, with practical implications for age-specific public health sleep recommendations.*

$$\eta^2 = \frac{SS_{\text{between}}}{SS_{\text{total}}}$$

In [98]:
# eta-squared: proportion of total variance explained by age group
eta_sq = SS_bw / SS_total
eta_sq

np.float64(0.9443655418972526)

In [99]:
# scipy verification
f_stat_sci, pval_sci = f_oneway(T, A, M, S)
f_stat_sci, pval_sci

(np.float64(90.53051643192494), np.float64(2.987331635366862e-10))

#### ✅ Final Conclusion

| Metric | Value |
|---|---|
| Test statistic (F) | 90.5305 |
| Critical value | 3.2389 |
| p-value | 2.99e-10 |
| df (between, within) | 3, 16 |
| α | 0.05 |
| η² | 0.9444 |
| Decision | **Reject H₀** |

At α = 0.05, there is strong evidence that age group significantly affects nightly sleep duration. The F statistic (90.53) far exceeds the critical value (3.24) and the p-value (2.99e-10) is effectively zero, so we reject H₀. η² = 0.9444 means age group explains approximately 94.4% of the total variance in sleep duration, an exceptionally large effect by Cohen's guidelines (large: η² > 0.14). The group means reveal a clear monotonic decline with age: Teens average roughly 7.88 hrs, Adults 7.0 hrs, Middle-aged 6.5 hrs, and Seniors 6.0 hrs. This pattern aligns with known physiological changes across the lifespan and underscores the importance of age-specific sleep guidelines in public health. Post-hoc analysis (e.g., Tukey HSD) would be needed to confirm which specific age group pairs differ significantly.